In [2]:
%pip install python-dotenv
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import sys
from datetime import date, datetime
from dotenv import load_dotenv
# Configuración de rutas
sys.path.append("../src")
# from repositories import *
from models import *



load_dotenv()

from domain.config import DB_URL
from repositories import UnitOfWorkFactory
from models import (
    Region, Player, Team, VideoGameType, 
    TournamentMode, TournamentStatus, Tournament, Match
)
from repositories import (
    PlayerRepository, TeamRepository, TournamentRepository, 
    RegionRepository, VideoGameTypeRepository
)

print(f"Entorno listo. Conectado a: {DB_URL}")

# Inicialización de la fábrica
uow_factory = UnitOfWorkFactory(DB_URL)

# La función de ayuda ahora llama a .create()
def get_uow():
    return uow_factory.create()

Entorno listo. Conectado a: sqlite:///:memory:


In [4]:
# --- FORZAR CREACIÓN DE TABLAS (Añadir al final de la Celda 1) ---
from domain.db import Base

# Usamos el engine de la factoría para crear lo que falte
Base.metadata.create_all(bind=uow_factory.engine)

print("Tablas verificadas/creadas en la base de datos.")

Tablas verificadas/creadas en la base de datos.


Adding Data For Testings

In [5]:
print("--- POBLANDO BASE DE DATOS CON TUS MODELOS Y REPOS ---")

with get_uow() as uow:
    # Usamos tus clases importadas directamente
    from repositories import (
        RegionRepository, VideoGameTypeRepository, TournamentModeRepository, 
        TournamentStatusRepository, PlayerRepository, TeamRepository, 
        TournamentRepository, MatchRepository
    )
    # Importamos también las intermedias y perfiles para limpieza
    from models import PlayerProfile, PlayerTeam, TeamTournament
    from datetime import datetime, date

    # 1. LIMPIEZA PREVIA (En orden para no romper Foreign Keys)
    # Esto permite que puedas ejecutar esta celda muchas veces sin errores de duplicados
    uow.session.query(Match).delete()
    uow.session.query(TeamTournament).delete()
    uow.session.query(PlayerTeam).delete()
    uow.session.query(PlayerProfile).delete()
    uow.session.query(Tournament).delete()
    uow.session.query(Player).delete()
    uow.session.query(Team).delete()
    uow.session.query(Region).delete()
    uow.session.query(VideoGameType).delete()
    uow.session.query(TournamentMode).delete()
    uow.session.query(TournamentStatus).delete()
    uow.commit()

    # 2. CREACIÓN DE CATÁLOGOS (Nivel 1: Sin dependencias)
    # Regiones
    reg_repo = RegionRepository(uow, Region)
    reg_eu = Region(name="Europa", country_code="EU")
    reg_na = Region(name="Norteamérica", country_code="NA")
    reg_repo.add(reg_eu)
    reg_repo.add(reg_na)

    # VideoGameTypes
    vgt_repo = VideoGameTypeRepository(uow, VideoGameType)
    vgt_moba = VideoGameType(name="MOBA", description="League of Legends, Dota 2")
    vgt_fps = VideoGameType(name="FPS", description="Counter Strike, Valorant")
    vgt_repo.add(vgt_moba)
    vgt_repo.add(vgt_fps)

    # Modos
    mode_repo = TournamentModeRepository(uow, TournamentMode)
    m_5v5 = TournamentMode(name="5vs5", description="Clásico por equipos")
    m_1v1 = TournamentMode(name="1vs1", description="Duelo de habilidad")
    mode_repo.add(m_5v5)
    mode_repo.add(m_1v1)

    # Estados
    status_repo = TournamentStatusRepository(uow, TournamentStatus)
    s_open = TournamentStatus(name="Abierto")
    s_live = TournamentStatus(name="En curso")
    status_repo.add(s_open)
    status_repo.add(s_live)
    
    uow.commit() # Guardamos para obtener los IDs generados

    # 3. ENTIDADES PRINCIPALES (Nivel 2: Dependen de los catálogos)
    # Equipos
    team_repo = TeamRepository(uow, Team)
    t1 = Team(name="Fnatic", creation_date=date(2004, 7, 23), id_region=reg_eu.id_region)
    t2 = Team(name="G2 Esports", creation_date=date(2013, 10, 15), id_region=reg_eu.id_region)
    team_repo.add(t1)
    team_repo.add(t2)

    # Jugadores
    player_repo = PlayerRepository(uow, Player)
    p1 = Player(
        nickname="Rekkles", email="rekkles@fnatic.com", 
        birth_date=datetime(1996, 9, 20), registration_date=datetime.now(), 
        id_region=reg_eu.id_region
    )
    p2 = Player(
        nickname="Caps", email="caps@g2.com", 
        birth_date=datetime(1999, 11, 17), registration_date=datetime.now(), 
        id_region=reg_eu.id_region
    )
    player_repo.add(p1)
    player_repo.add(p2)
    
    uow.commit()

    # 4. PERFILES Y RELACIONES (Nivel 3: Relaciones N a M y Perfiles)
    # Perfiles (PlayerProfile)
    uow.session.add(PlayerProfile(id_player=p1.id_player, ranking_position=5, total_matches=150))
    uow.session.add(PlayerProfile(id_player=p2.id_player, ranking_position=1, total_matches=300))

    # Vínculos Jugador-Equipo (PlayerTeam) - Usando tu campo 'registered_at'
    uow.session.add(PlayerTeam(id_player=p1.id_player, id_team=t1.id_team, registered_at=datetime.now()))
    uow.session.add(PlayerTeam(id_player=p2.id_player, id_team=t2.id_team, registered_at=datetime.now()))
    
    uow.commit()

    # 5. TORNEOS Y PARTIDOS (Nivel 4: Competición)
    tour_repo = TournamentRepository(uow, Tournament)
    lec = Tournament(
        name="LEC Summer 2026",
        start_date=date(2026, 6, 1),
        end_date=date(2026, 8, 30),
        id_region=reg_eu.id_region,
        id_video_game_type=vgt_moba.id_video_game_type,
        id_tournament_mode=m_5v5.id_tournament_mode,
        id_tournament_status=s_live.id_tournament_status,
        prize_pool=200000.0,
        max_teams=10
    )
    tour_repo.add(lec)
    uow.commit()

    # Partidos (Match) - Usando tus campos 'id_team_one' e 'id_team_two'
    match_repo = MatchRepository(uow, Match)
    m1 = Match(
        id_tournament=lec.id_tournament,
        id_team_one=t1.id_team,
        id_team_two=t2.id_team,
        id_team_winner=t2.id_team,
        score_team_one=0,
        score_team_two=2,
        round_number=1,
        duration_minutes=42,
        match_date=datetime.now()
    )
    match_repo.add(m1)
    uow.commit()

print("✅ Datos temporales cargados. Ya puedes realizar tus pruebas tranquilamente.")

--- POBLANDO BASE DE DATOS CON TUS MODELOS Y REPOS ---
✅ Datos temporales cargados. Ya puedes realizar tus pruebas tranquilamente.


Player

In [6]:
print("--- VERIFICANDO FUNCIONES DE CONSULTA EN PlayerRepository ---")

with get_uow() as uow:
    from repositories import PlayerRepository
    from models import Player
    
    # Instanciamos el repositorio con el UOW actual
    player_repo = PlayerRepository(uow, Player)

    # 1. Prueba de get_by_nickname
    print("\n[TEST] get_by_nickname('Faker')")
    faker = player_repo.get_by_nickname("Faker")
    if faker:
        print(f"✅ Éxito: Encontrado ID {faker.id_player} con email {faker.email}")
    else:
        print("❌ Error: No se encontró a Faker.")

    # 2. Prueba de get_paginated
    print("\n[TEST] get_paginated(page=1, page_size=2)")
    pagina = player_repo.get_paginated(page=1, page_size=2)
    print(f"✅ Éxito: Se recuperaron {len(pagina)} jugadores de la página 1.")
    for p in pagina:
        print(f"   - {p.nickname}")

    # 3. Prueba de get_players_by_region (Región 1: Europa)
    # En el script anterior pusimos a Xpeke y S1mple en la región 1
    print("\n[TEST] get_players_by_region(region_id=1)")
    europeos = player_repo.get_players_by_region(region_id=1)
    print(f"✅ Éxito: {len(europeos)} jugadores encontrados en Europa.")
    for p in europeos:
        print(f"   - {p.nickname}")

    # 4. Prueba de get_global_ranking
    # Faker (Rank 1), S1mple (Rank 2), Xpeke (Rank 5)
    print("\n[TEST] get_global_ranking(limit=3)")
    ranking = player_repo.get_global_ranking(limit=3)
    if len(ranking) > 0:
        print(f"✅ Éxito: Ranking recuperado.")
        for i, p in enumerate(ranking, 1):
            # Accedemos al profile vinculado para ver la posición real
            print(f"   {i}. {p.nickname} (Posición en DB: {p.profile.ranking_position})")
    else:
        print("❌ Error: El ranking volvió vacío. ¿Se cargaron los PlayerProfile?")

    # 5. Prueba de get_player_team_history (Player 1: Xpeke)
    print("\n[TEST] get_player_team_history(player_id=1)")
    historial = player_repo.get_player_team_history(player_id=1)
    if len(historial) > 0:
        print(f"✅ Éxito: Se encontraron {len(historial)} registros históricos.")
        for registro in historial:
            print(f"   - Equipo ID: {registro.id_team} | Fecha registro: {registro.registered_at}")
    else:
        print("❌ Error: No se encontró historial para el jugador 1.")

print("\n--- PRUEBAS DE LECTURA FINALIZADAS ---")

--- VERIFICANDO FUNCIONES DE CONSULTA EN PlayerRepository ---

[TEST] get_by_nickname('Faker')
❌ Error: No se encontró a Faker.

[TEST] get_paginated(page=1, page_size=2)
✅ Éxito: Se recuperaron 2 jugadores de la página 1.
   - Rekkles
   - Caps

[TEST] get_players_by_region(region_id=1)
✅ Éxito: 2 jugadores encontrados en Europa.
   - Rekkles
   - Caps

[TEST] get_global_ranking(limit=3)
✅ Éxito: Ranking recuperado.
   1. Caps (Posición en DB: 1)
   2. Rekkles (Posición en DB: 5)

[TEST] get_player_team_history(player_id=1)
✅ Éxito: Se encontraron 1 registros históricos.
   - Equipo ID: 1 | Fecha registro: 2026-05-18 16:19:33.375376

--- PRUEBAS DE LECTURA FINALIZADAS ---


Match

In [7]:
print("--- VERIFICANDO FUNCIONES DE MatchRepository ---")

with get_uow() as uow:
    from repositories import MatchRepository
    from models import Match
    
    match_repo = MatchRepository(uow, Match)

    # 1. Prueba de get_matches_by_tournament
    # Tenemos cargado el torneo ID 1 (Worlds 2026)
    print("\n[TEST] get_matches_by_tournament(tournament_id=1)")
    matches_tour = match_repo.get_matches_by_tournament(tournament_id=1)
    if len(matches_tour) > 0:
        print(f"✅ Éxito: Se encontraron {len(matches_tour)} partidos en el torneo.")
        for m in matches_tour:
            print(f"   - Match ID: {m.id_match} | Fecha: {m.match_date}")
    else:
        print("❌ Error: No se encontraron partidos para el torneo 1.")

    # 2. Prueba de get_head_to_head
    # Tenemos a Fnatic (ID 1) vs T1 (ID 2)
    print("\n[TEST] get_head_to_head(team_a=1, team_b=2)")
    h2h = match_repo.get_head_to_head(team_a_id=1, team_b_id=2)
    if len(h2h) > 0:
        print(f"✅ Éxito: {len(h2h)} enfrentamiento(s) directo(s) encontrado(s).")
        for m in h2h:
            print(f"   - {m.match_date}: Equipo {m.id_team_one} vs Equipo {m.id_team_two}")
    else:
        print("❌ Error: No se encontró el historial entre los equipos 1 y 2.")

    # 3. Prueba de get_recent_results
    # El partido que cargamos tiene un ganador (T1), así que debe aparecer aquí
    print("\n[TEST] get_recent_results(limit=10)")
    recientes = match_repo.get_recent_results(limit=10)
    if len(recientes) > 0:
        print(f"✅ Éxito: Se recuperaron {len(recientes)} resultados recientes.")
        for m in recientes:
            print(f"   - Ganador: Equipo {m.id_team_winner} | Marcador: {m.score_team_one}-{m.score_team_two}")
    else:
        print("❌ Error: No hay resultados recientes. ¿Tienen los partidos un 'id_team_winner'?")

    # 4. Prueba de get_tournament_bracket
    # Ordenado por ronda
    print("\n[TEST] get_tournament_bracket(tournament_id=1)")
    bracket = match_repo.get_tournament_bracket(tournament_id=1)
    if len(bracket) > 0:
        print(f"✅ Éxito: Bracket generado con {len(bracket)} partidos.")
        for m in bracket:
            print(f"   - Ronda {m.round_number}: Match {m.id_match}")
    else:
        print("❌ Error: El bracket está vacío.")

print("\n--- PRUEBAS DE MATCH FINALIZADAS ---")

--- VERIFICANDO FUNCIONES DE MatchRepository ---

[TEST] get_matches_by_tournament(tournament_id=1)
✅ Éxito: Se encontraron 1 partidos en el torneo.
   - Match ID: 1 | Fecha: 2026-05-18 16:19:33.377378

[TEST] get_head_to_head(team_a=1, team_b=2)
✅ Éxito: 1 enfrentamiento(s) directo(s) encontrado(s).
   - 2026-05-18 16:19:33.377378: Equipo 1 vs Equipo 2

[TEST] get_recent_results(limit=10)
✅ Éxito: Se recuperaron 1 resultados recientes.
   - Ganador: Equipo 2 | Marcador: 0-2

[TEST] get_tournament_bracket(tournament_id=1)
✅ Éxito: Bracket generado con 1 partidos.
   - Ronda 1: Match 1

--- PRUEBAS DE MATCH FINALIZADAS ---


Player Profile

In [8]:
print("--- VERIFICANDO FUNCIONES DE PlayerProfileRepository ---")

with get_uow() as uow:
    from repositories import PlayerProfileRepository
    from models import PlayerProfile
    
    # Instanciamos el repositorio
    profile_repo = PlayerProfileRepository(uow, PlayerProfile)

    # 1. Prueba de get_by_player_id
    # Buscamos el perfil del jugador 2 (Faker)
    print("\n[TEST] get_by_player_id(player_id=2)")
    perfil_faker = profile_repo.get_by_player_id(player_id=2)
    if perfil_faker:
        print(f"✅ Éxito: Perfil encontrado.")
        print(f"   - Partidas totales: {perfil_faker.total_matches}")
        print(f"   - Victorias: {perfil_faker.total_wins}")
    else:
        print("❌ Error: No se encontró el perfil para el jugador 2.")

    # 2. Prueba de get_top_ranked
    # Debería devolver: Faker (1), S1mple (2), Xpeke (5)
    print("\n[TEST] get_top_ranked(limit=3)")
    top_perfiles = profile_repo.get_top_ranked(limit=3)
    if len(top_perfiles) > 0:
        print(f"✅ Éxito: Se recuperaron {len(top_perfiles)} perfiles ordenados por ranking.")
        for i, p in enumerate(top_perfiles, 1):
            # Nota: El objeto PlayerProfile tiene el id_player, 
            # si quieres el nombre deberías acceder a p.player.nickname
            print(f"   {i}. Jugador ID: {p.id_player} | Posición: {p.ranking_position}")
    else:
        print("❌ Error: La lista de ranking está vacía.")

    # 3. Prueba de get_most_active
    # Debería ordenar por total_matches de mayor a menor
    print("\n[TEST] get_most_active(limit=3)")
    activos = profile_repo.get_most_active(limit=3)
    if len(activos) > 0:
        print(f"✅ Éxito: Se recuperaron los perfiles más activos.")
        for p in activos:
            print(f"   - Jugador ID: {p.id_player} | Partidas jugadas: {p.total_matches}")
        
        # Verificación lógica de orden descendente
        if len(activos) >= 2 and activos[0].total_matches >= activos[1].total_matches:
            print("   (El orden descendente es correcto)")
    else:
        print("❌ Error: No se recuperaron perfiles activos.")

print("\n--- PRUEBAS DE PLAYER PROFILE FINALIZADAS ---")

--- VERIFICANDO FUNCIONES DE PlayerProfileRepository ---

[TEST] get_by_player_id(player_id=2)
✅ Éxito: Perfil encontrado.
   - Partidas totales: 300
   - Victorias: 0

[TEST] get_top_ranked(limit=3)
✅ Éxito: Se recuperaron 2 perfiles ordenados por ranking.
   1. Jugador ID: 2 | Posición: 1
   2. Jugador ID: 1 | Posición: 5

[TEST] get_most_active(limit=3)
✅ Éxito: Se recuperaron los perfiles más activos.
   - Jugador ID: 2 | Partidas jugadas: 300
   - Jugador ID: 1 | Partidas jugadas: 150
   (El orden descendente es correcto)

--- PRUEBAS DE PLAYER PROFILE FINALIZADAS ---


Player Team

In [9]:
print("--- VERIFICANDO FUNCIONES DE PlayerTeamRepository ---")

with get_uow() as uow:
    from repositories import PlayerTeamRepository
    from models import PlayerTeam
    
    # Instanciamos el repositorio
    pt_repo = PlayerTeamRepository(uow, PlayerTeam)

    # 1. Prueba de list()
    # Esta usa el estilo clásico query().all(), debería funcionar
    print("\n[TEST] list()")
    todos = pt_repo.list()
    print(f"✅ Éxito: Se encontraron {len(todos)} vínculos totales en la tabla intermedia.")

    # 2. Prueba de is_player_in_team
    # Verificamos si Xpeke (1) está en Fnatic (1)
    print("\n[TEST] is_player_in_team(player=1, team=1)")
    esta_dentro = pt_repo.is_player_in_team(player_id=1, team_id=1)
    if esta_dentro:
        print(f"✅ Éxito: El sistema confirma que el jugador 1 está en el equipo 1.")
    else:
        print("❌ Error: No se detectó el vínculo (o no se cargó en el paso anterior).")

    # 3. Pruebas que podrían fallar por el nombre del campo (join_date vs registered_at)
    print("\n[TEST] Comprobando funciones con ordenación por fecha...")
    
    # Prueba: list_players_in_team
    try:
        roster = pt_repo.list_players_in_team(team_id=1)
        print(f"✅ Éxito: El equipo 1 tiene {len(roster)} jugadores.")
    except AttributeError as e:
        print(f"❌ Error en list_players_in_team: {e}")
        print("   (Sugerencia: Cambia 'join_date' por 'registered_at' en el .py del repositorio)")

    # Prueba: get_current_team
    try:
        actual = pt_repo.get_current_team(player_id=1)
        if actual:
            print(f"✅ Éxito: El equipo actual del jugador 1 es el ID {actual.id_team}.")
    except AttributeError as e:
        print(f"❌ Error en get_current_team: {e}")
        print("   (Sugerencia: Cambia 'join_date' por 'registered_at' en el .py del repositorio)")

print("\n--- PRUEBAS DE PLAYER TEAM FINALIZADAS ---")

--- VERIFICANDO FUNCIONES DE PlayerTeamRepository ---

[TEST] list()
✅ Éxito: Se encontraron 2 vínculos totales en la tabla intermedia.

[TEST] is_player_in_team(player=1, team=1)
✅ Éxito: El sistema confirma que el jugador 1 está en el equipo 1.

[TEST] Comprobando funciones con ordenación por fecha...
✅ Éxito: El equipo 1 tiene 1 jugadores.
✅ Éxito: El equipo actual del jugador 1 es el ID 1.

--- PRUEBAS DE PLAYER TEAM FINALIZADAS ---


Region

In [10]:
print("--- VERIFICANDO FUNCIONES DE RegionRepository ---")

with get_uow() as uow:
    from repositories import RegionRepository
    from models import Region
    
    # Instanciamos el repositorio
    reg_repo = RegionRepository(uow, Region)

    # 1. Prueba de get_by_country_code
    # Buscamos la región de Europa por su código 'EU'
    print("\n[TEST] get_by_country_code('EU')")
    region_eu = reg_repo.get_by_country_code(code="EU")
    if region_eu:
        print(f"✅ Éxito: Encontrada región '{region_eu.name}' con ID {region_eu.id_region}")
    else:
        print("❌ Error: No se encontró la región con código 'EU'.")

    # 2. Prueba de count_players
    # En la carga anterior pusimos a Xpeke (1) y S1mple (3) en la región 1 (Europa)
    print("\n[TEST] count_players(region_id=1)")
    total_players = reg_repo.count_players(region_id=1)
    if total_players == 2:
        print(f"✅ Éxito: Conteo exacto ({total_players} jugadores).")
    else:
        print(f"⚠️ Aviso: Se contaron {total_players} jugadores (Se esperaban 2 si usaste la carga previa).")

    # 3. Prueba de list_with_tournaments
    # En la carga anterior, solo el torneo 'Worlds 2026' está en la región 3 (Asia)
    print("\n[TEST] list_with_tournaments()")
    regiones_activas = reg_repo.list_with_tournaments()
    if len(regiones_activas) > 0:
        print(f"✅ Éxito: Se encontraron {len(regiones_activas)} regiones con torneos.")
        for r in regiones_activas:
            print(f"   - Región con competición: {r.name} (ID {r.id_region})")
    else:
        print("❌ Error: No se detectaron regiones con torneos activos.")

print("\n--- PRUEBAS DE REGIÓN FINALIZADAS ---")

--- VERIFICANDO FUNCIONES DE RegionRepository ---

[TEST] get_by_country_code('EU')
✅ Éxito: Encontrada región 'Europa' con ID 1

[TEST] count_players(region_id=1)
✅ Éxito: Conteo exacto (2 jugadores).

[TEST] list_with_tournaments()
✅ Éxito: Se encontraron 1 regiones con torneos.
   - Región con competición: Europa (ID 1)

--- PRUEBAS DE REGIÓN FINALIZADAS ---


Team

In [11]:
print("--- INSCRIBIENDO EQUIPOS EN EL TORNEO ---")

with get_uow() as uow:
    from models import TeamTournament
    from datetime import datetime

    # Inscribimos a Fnatic (ID 1) y G2 (ID 2) en el torneo Worlds (ID 1)
    # Importante: usamos 'registered_at' que es el nombre de tu modelo
    uow.session.add(TeamTournament(id_team=1, id_tournament=1, points_obtained=10, registered_at=datetime.now()))
    uow.session.add(TeamTournament(id_team=2, id_tournament=1, points_obtained=25, registered_at=datetime.now()))
    
    uow.commit()

print("✅ Equipos inscritos. ¡Vuelve a ejecutar el test del TeamRepository!")



print("--- VERIFICANDO FUNCIONES DE TeamRepository ---")

with get_uow() as uow:
    from repositories import TeamRepository
    from models import Team
    
    # Instanciamos el repositorio
    team_repo = TeamRepository(uow, Team)

    # 1. Prueba de get_full_roster (Eager Loading)
    # Buscamos Fnatic (ID 1), que debería tener a Xpeke vinculado
    print("\n[TEST] get_full_roster(team_id=1)")
    team_full = team_repo.get_full_roster(team_id=1)
    if team_full:
        print(f"✅ Éxito: Equipo '{team_full.name}' recuperado con sus relaciones.")
        # Verificamos si los jugadores se cargaron gracias al joinedload
        if team_full.team_players:
            print(f"   - Jugadores detectados en el roster: {len(team_full.team_players)}")
            for pt in team_full.team_players:
                # Al usar joinedload en el repo, esto no dispara nuevas consultas
                print(f"     * Jugador: {pt.player.nickname} (Unido el: {pt.registered_at})")
        else:
            print("   ⚠️  Aviso: El equipo no tiene jugadores vinculados.")
    else:
        print("❌ Error: No se encontró el equipo 1.")

    # 2. Prueba de get_teams_by_game_type
    # Buscamos equipos que hayan participado en juegos tipo MOBA (ID 1)
    print("\n[TEST] get_teams_by_game_type(game_type_id=1)")
    moba_teams = team_repo.get_teams_by_game_type(game_type_id=1)
    if len(moba_teams) > 0:
        print(f"✅ Éxito: Se encontraron {len(moba_teams)} equipos de MOBA.")
        for t in moba_teams:
            print(f"   - {t.name}")
    else:
        print("❌ Error: No se encontraron equipos para el tipo de juego 1.")

    # 3. Prueba de get_win_loss_ratio
    # Según nuestro seeder, T1 (ID 2) ganó el partido contra Fnatic (ID 1)
    print("\n[TEST] get_win_loss_ratio(team_id=2) - Caso T1")
    stats_t1 = team_repo.get_win_loss_ratio(team_id=2)
    print(f"✅ Éxito: Estadísticas de T1 recuperadas.")
    print(f"   - Victorias: {stats_t1['wins']} | Derrotas: {stats_t1['losses']}")
    print(f"   - Total Partidos: {stats_t1['total_matches']} | Ratio W/L: {stats_t1['win_ratio']}")

    print("\n[TEST] get_win_loss_ratio(team_id=1) - Caso Fnatic")
    stats_f1 = team_repo.get_win_loss_ratio(team_id=1)
    print(f"✅ Éxito: Estadísticas de Fnatic recuperadas.")
    print(f"   - Victorias: {stats_f1['wins']} | Derrotas: {stats_f1['losses']}")
    print(f"   - Ratio W/L: {stats_f1['win_ratio']}")

print("\n--- PRUEBAS DE TEAM FINALIZADAS ---")

--- INSCRIBIENDO EQUIPOS EN EL TORNEO ---
✅ Equipos inscritos. ¡Vuelve a ejecutar el test del TeamRepository!
--- VERIFICANDO FUNCIONES DE TeamRepository ---

[TEST] get_full_roster(team_id=1)
✅ Éxito: Equipo 'Fnatic' recuperado con sus relaciones.
   - Jugadores detectados en el roster: 1
     * Jugador: Rekkles (Unido el: 2026-05-18 16:19:33.375376)

[TEST] get_teams_by_game_type(game_type_id=1)
✅ Éxito: Se encontraron 2 equipos de MOBA.
   - Fnatic
   - G2 Esports

[TEST] get_win_loss_ratio(team_id=2) - Caso T1
✅ Éxito: Estadísticas de T1 recuperadas.
   - Victorias: 1 | Derrotas: 0
   - Total Partidos: 1 | Ratio W/L: 1.0

[TEST] get_win_loss_ratio(team_id=1) - Caso Fnatic
✅ Éxito: Estadísticas de Fnatic recuperadas.
   - Victorias: 0 | Derrotas: 1
   - Ratio W/L: 0.0

--- PRUEBAS DE TEAM FINALIZADAS ---


Team Tournament

In [12]:
print("--- VERIFICANDO FUNCIONES DE TeamTournamentRepository ---")

with get_uow() as uow:
    from repositories import TeamTournamentRepository
    from models import TeamTournament
    
    # Instanciamos el repositorio
    tt_repo = TeamTournamentRepository(uow, TeamTournament)

    # 1. Prueba de get_registration
    # Buscamos la inscripción de Fnatic (ID 1) en el torneo Worlds (ID 1)
    print("\n[TEST] get_registration(team=1, tournament=1)")
    reg = tt_repo.get_registration(team_id=1, tournament_id=1)
    if reg:
        print(f"✅ Éxito: Registro encontrado.")
        print(f"   - Puntos actuales: {reg.points_obtained}")
        print(f"   - Fecha de inscripción: {reg.registered_at}")
    else:
        print("❌ Error: No se encontró la inscripción del equipo 1 en el torneo 1.")

    # 2. Prueba de list_registered_teams
    # Debería devolver la clasificación del torneo ID 1
    print("\n[TEST] list_registered_teams(tournament=1)")
    leaderboard = tt_repo.list_registered_teams(tournament_id=1)
    if len(leaderboard) > 0:
        print(f"✅ Éxito: Se recuperaron {len(leaderboard)} equipos inscritos.")
        for i, entry in enumerate(leaderboard, 1):
            # Accedemos a entry.team.name gracias a la relación en el modelo
            print(f"   {i}. Equipo ID: {entry.id_team} | Puntos: {entry.points_obtained}")
    else:
        print("❌ Error: No hay equipos inscritos en este torneo.")

    # 3. Prueba de update_points (Lógica en sesión)
    # Vamos a sumar 3 puntos a Fnatic (ID 1)
    print("\n[TEST] update_points(team=1, tournament=1, points=3)")
    puntos_antes = reg.points_obtained if reg else 0
    updated_reg = tt_repo.update_points(team_id=1, tournament_id=1, points_to_add=3)
    
    if updated_reg and updated_reg.points_obtained == (puntos_antes + 3):
        print(f"✅ Éxito: Puntos actualizados en memoria ({puntos_antes} -> {updated_reg.points_obtained}).")
        # No hacemos uow.commit() para no alterar permanentemente los datos si no quieres
        print("   (Los cambios están en la sesión, listos para commit)")
    else:
        print("❌ Error: La actualización de puntos falló.")

print("\n--- PRUEBAS DE TEAM TOURNAMENT FINALIZADAS ---")

--- VERIFICANDO FUNCIONES DE TeamTournamentRepository ---

[TEST] get_registration(team=1, tournament=1)
✅ Éxito: Registro encontrado.
   - Puntos actuales: 10
   - Fecha de inscripción: 2026-05-18 16:19:33.470036

[TEST] list_registered_teams(tournament=1)
✅ Éxito: Se recuperaron 2 equipos inscritos.
   1. Equipo ID: 2 | Puntos: 25
   2. Equipo ID: 1 | Puntos: 10

[TEST] update_points(team=1, tournament=1, points=3)
✅ Éxito: Puntos actualizados en memoria (10 -> 13).
   (Los cambios están en la sesión, listos para commit)

--- PRUEBAS DE TEAM TOURNAMENT FINALIZADAS ---


Tournament

In [13]:
print("--- VERIFICANDO FUNCIONES DE TournamentRepository ---")

with get_uow() as uow:
    from repositories import TournamentRepository
    from models import Tournament, Match
    from datetime import datetime

    # Instanciamos el repositorio
    tour_repo = TournamentRepository(uow, Tournament)

    # 1. Prueba de get_active_tournaments
    # El torneo 'Worlds 2026' lo creamos con el estado 'En curso'
    print("\n[TEST] get_active_tournaments(status_name='En curso')")
    activos = tour_repo.get_active_tournaments(status_name="En curso")
    if len(activos) > 0:
        print(f"✅ Éxito: Se encontraron {len(activos)} torneos activos.")
        for t in activos:
            print(f"   - Torneo: {t.name} | Inicio: {t.start_date}")
    else:
        print("❌ Error: No se detectaron torneos 'En curso'. Revisa los nombres en tu tabla TournamentStatus.")

    # 2. Prueba de get_leaderboard
    # Debería mostrar la clasificación del torneo Worlds (ID 1)
    print("\n[TEST] get_leaderboard(tournament_id=1)")
    leaderboard = tour_repo.get_leaderboard(tournament_id=1)
    if len(leaderboard) > 0:
        print(f"✅ Éxito: Clasificación recuperada.")
        for entry in leaderboard:
            # Gracias al joinedload(TeamTournament.team), accedemos al nombre sin otra consulta
            print(f"   - Equipo: {entry.team.name} | Puntos: {entry.points_obtained}")
    else:
        print("❌ Error: No se pudo generar la clasificación para el torneo 1.")

    # 3. Prueba de get_top_players
    # Cruza Jugadores -> Equipos -> Puntos del Torneo
    print("\n[TEST] get_top_players(tournament_id=1)")
    top_players = tour_repo.get_top_players(tournament_id=1, limit=5)
    if len(top_players) > 0:
        print(f"✅ Éxito: Se recuperaron {len(top_players)} jugadores destacados.")
        for p in top_players:
            print(f"   - Jugador: {p.nickname}")
    else:
        print("❌ Error: No se encontraron jugadores vinculados a este torneo.")

    # 4. Prueba de get_upcoming_matches
    # Primero creamos un partido rápido sin ganador para probar la función
    print("\n[TEST] get_upcoming_matches(tournament_id=1)")
    
    # Añadimos un partido pendiente solo para el test si no existe ninguno
    proximo_match = Match(
        id_tournament=1, id_team_one=1, id_team_two=2, 
        round_number=2, duration_minutes=0,
        match_date=datetime(2026, 11, 15, 20, 0)
    )
    uow.session.add(proximo_match)
    uow.commit() # Guardamos el partido pendiente

    proximos = tour_repo.get_upcoming_matches(tournament_id=1)
    if len(proximos) > 0:
        print(f"✅ Éxito: Se encontraron {len(proximos)} partidos pendientes.")
        for m in proximos:
            print(f"   - Fecha: {m.match_date} | Ronda: {m.round_number} (Sin ganador aún)")
    else:
        print("❌ Error: No se detectaron partidos pendientes.")

print("\n--- PRUEBAS DE TOURNAMENT FINALIZADAS ---")

--- VERIFICANDO FUNCIONES DE TournamentRepository ---

[TEST] get_active_tournaments(status_name='En curso')
✅ Éxito: Se encontraron 1 torneos activos.
   - Torneo: LEC Summer 2026 | Inicio: 2026-06-01

[TEST] get_leaderboard(tournament_id=1)
✅ Éxito: Clasificación recuperada.
   - Equipo: G2 Esports | Puntos: 25
   - Equipo: Fnatic | Puntos: 13

[TEST] get_top_players(tournament_id=1)
✅ Éxito: Se recuperaron 2 jugadores destacados.
   - Jugador: Caps
   - Jugador: Rekkles

[TEST] get_upcoming_matches(tournament_id=1)
✅ Éxito: Se encontraron 1 partidos pendientes.
   - Fecha: 2026-11-15 20:00:00 | Ronda: 2 (Sin ganador aún)

--- PRUEBAS DE TOURNAMENT FINALIZADAS ---


T - Mode

In [14]:
print("--- VERIFICANDO FUNCIONES DE TournamentModeRepository ---")

with get_uow() as uow:
    from repositories import TournamentModeRepository
    from models import TournamentMode
    
    # Instanciamos el repositorio
    mode_repo = TournamentModeRepository(uow, TournamentMode)

    # 1. Prueba de get_by_name
    # Probamos con "5vs5". Usamos ilike, así que debería dar igual "5VS5" o "5vs5"
    print("\n[TEST] get_by_name('5vs5')")
    modo = mode_repo.get_by_name("5vs5")
    if modo:
        print(f"✅ Éxito: Modo encontrado.")
        print(f"   - Nombre: {modo.name}")
        print(f"   - Descripción: {modo.description}")
    else:
        print("❌ Error: No se encontró el modo '5vs5'.")

    # 2. Prueba de list_active_modes
    # Debería devolver todos los modos que insertamos (5vs5 y 1vs1)
    print("\n[TEST] list_active_modes()")
    modos = mode_repo.list_active_modes()
    if len(modos) > 0:
        print(f"✅ Éxito: Se recuperaron {len(modos)} modos de torneo.")
        for m in modos:
            print(f"   - ID {m.id_tournament_mode}: {m.name}")
    else:
        print("❌ Error: La lista de modos está vacía.")

print("\n--- PRUEBAS DE TOURNAMENT MODE FINALIZADAS ---")

--- VERIFICANDO FUNCIONES DE TournamentModeRepository ---

[TEST] get_by_name('5vs5')
✅ Éxito: Modo encontrado.
   - Nombre: 5vs5
   - Descripción: Clásico por equipos

[TEST] list_active_modes()
✅ Éxito: Se recuperaron 2 modos de torneo.
   - ID 1: 5vs5
   - ID 2: 1vs1

--- PRUEBAS DE TOURNAMENT MODE FINALIZADAS ---


T - Status

In [15]:
print("--- VERIFICANDO FUNCIONES DE TournamentStatusRepository ---")

with get_uow() as uow:
    from repositories import TournamentStatusRepository
    from models import TournamentStatus
    
    # Instanciamos el repositorio
    status_repo = TournamentStatusRepository(uow, TournamentStatus)

    # 1. Prueba de get_by_name ('En curso')
    print("\n[TEST] get_by_name('En curso')")
    st_live = status_repo.get_by_name("En curso")
    if st_live:
        print(f"✅ Éxito: Estado encontrado.")
        print(f"   - ID: {st_live.id_tournament_status} | Nombre: {st_live.name}")
    else:
        print("❌ Error: No se encontró el estado 'En curso'.")

    # 2. Prueba de get_by_name ('Abierto')
    print("\n[TEST] get_by_name('Abierto')")
    st_open = status_repo.get_by_name("Abierto")
    if st_open:
        print(f"✅ Éxito: Estado encontrado.")
        print(f"   - ID: {st_open.id_tournament_status} | Nombre: {st_open.name}")
    else:
        print("❌ Error: No se encontró el estado 'Abierto'.")

    # 3. Prueba de get_all (Heredada del repositorio base)
    print("\n[TEST] get_all()")
    todos = status_repo.get_all()
    if len(todos) > 0:
        print(f"✅ Éxito: Se recuperaron {len(todos)} estados posibles.")
        for s in todos:
            print(f"   - {s.name} (ID: {s.id_tournament_status})")
    else:
        print("❌ Error: No se encontraron estados en la tabla.")

print("\n--- PRUEBAS DE TOURNAMENT STATUS FINALIZADAS ---")

--- VERIFICANDO FUNCIONES DE TournamentStatusRepository ---

[TEST] get_by_name('En curso')
✅ Éxito: Estado encontrado.
   - ID: 2 | Nombre: En curso

[TEST] get_by_name('Abierto')
✅ Éxito: Estado encontrado.
   - ID: 1 | Nombre: Abierto

[TEST] get_all()
✅ Éxito: Se recuperaron 2 estados posibles.
   - Abierto (ID: 1)
   - En curso (ID: 2)

--- PRUEBAS DE TOURNAMENT STATUS FINALIZADAS ---


VideoGame Type

In [16]:
print("--- VERIFICANDO FUNCIONES DE VideoGameTypeRepository ---")

with get_uow() as uow:
    from repositories import VideoGameTypeRepository
    from models import VideoGameType
    
    # Instanciamos el repositorio
    vgt_repo = VideoGameTypeRepository(uow, VideoGameType)

    # 1. Prueba de get_by_name ('MOBA' exacto)
    print("\n[TEST] get_by_name('MOBA')")
    moba = vgt_repo.get_by_name("MOBA")
    if moba:
        print(f"✅ Éxito: Tipo de juego encontrado.")
        print(f"   - ID: {moba.id_video_game_type} | Nombre: {moba.name}")
        print(f"   - Descripción: {moba.description}")
    else:
        print("❌ Error: No se encontró el tipo 'MOBA'.")

    # 2. Prueba de get_by_name con insensibilidad a mayúsculas ('moba')
    print("\n[TEST] get_by_name('moba') - Probando ILIKE")
    moba_lower = vgt_repo.get_by_name("moba")
    if moba_lower and moba_lower.name == "MOBA":
        print(f"✅ Éxito: ILIKE funcionó. 'moba' devolvió el registro 'MOBA'.")
    else:
        print("❌ Error: La búsqueda insensible a mayúsculas falló.")

    # 3. Prueba de get_all (Heredada)
    print("\n[TEST] get_all()")
    tipos = vgt_repo.get_all()
    if len(tipos) > 0:
        print(f"✅ Éxito: Se recuperaron {len(tipos)} tipos de videojuegos.")
        for t in tipos:
            print(f"   - {t.name} (ID: {t.id_video_game_type})")
    else:
        print("❌ Error: No hay tipos de juego registrados.")

print("\n--- PRUEBAS DE VIDEO GAME TYPE FINALIZADAS ---")

--- VERIFICANDO FUNCIONES DE VideoGameTypeRepository ---

[TEST] get_by_name('MOBA')
✅ Éxito: Tipo de juego encontrado.
   - ID: 1 | Nombre: MOBA
   - Descripción: League of Legends, Dota 2

[TEST] get_by_name('moba') - Probando ILIKE
✅ Éxito: ILIKE funcionó. 'moba' devolvió el registro 'MOBA'.

[TEST] get_all()
✅ Éxito: Se recuperaron 2 tipos de videojuegos.
   - MOBA (ID: 1)
   - FPS (ID: 2)

--- PRUEBAS DE VIDEO GAME TYPE FINALIZADAS ---
